# Causal GPT-RL — Training Quickstart

Launch a Causal GPT-RL training job on SageMaker, load policy bundles while the
job is still running, and open the finished model with the public runtime.

Subscribe first, on AWS Marketplace:
<https://aws.amazon.com/marketplace/pp/prodview-is6jt3bcwkq5c>

**What you need before running this**

- An active subscription to the training algorithm.
- A SageMaker execution role that can read your data prefix and write your
  output and checkpoint prefixes.
- Three S3 prefixes — data, output artifact, checkpoints — **all in the region
  you launch the job in**. AWS requires this of the training input bucket and of
  the checkpoint prefix; the output prefix joins them to keep the setup to one
  region and off cross-region transfer charges.
- A dataset. Section 2 fetches a public one and uploads it to your bucket, so
  you do not need to bring your own to get through this notebook.

This notebook is standalone. It installs what it needs and imports nothing from
the repository it lives in.

Reference docs:
[running a job](https://github.com/ccnets-team/causal-gpt-rl/blob/main/training/docs/aws/aws-marketplace-training.md) ·
[training inputs](https://github.com/ccnets-team/causal-gpt-rl/blob/main/training/docs/aws/sagemaker-inputs.md) ·
[checkpoints and bundles](https://github.com/ccnets-team/causal-gpt-rl/blob/main/training/docs/aws/sagemaker-checkpoints.md)

In [ ]:
# The `<3` bound is load-bearing: AlgorithmEstimator does not exist in the
# SageMaker SDK v3, so an unbounded install breaks section 4.
# `minari` is used in section 2, to fetch the quickstart dataset. `causal-gpt-rl`
# is used by the last two sections, which open a bundle; it pulls in torch, so
# drop it if all you want is to launch the job.
%pip install -q "sagemaker>=2.200,<3" boto3 "minari==0.5.3" causal-gpt-rl

## 1. Configure

The Algorithm ARN comes from the **Configure** screen of the listing, after your
subscription is active. Pick your region there and copy the ARN it shows.

Do not assemble the ARN yourself. The account inside it is a Marketplace account
that differs per region, and the algorithm name is not the product version — the
only correct value is the one on that screen for the region you are launching in.

In [ ]:
import boto3
import sagemaker

# --- From the Marketplace listing, for the region you are launching in ---
ALGORITHM_ARN = "arn:aws:sagemaker:<region>:<marketplace-account>:algorithm/<name-and-hash>"

# --- Yours ---
ROLE_ARN = "arn:aws:iam::<your-account>:role/<your-sagemaker-execution-role>"
DATA_S3 = "s3://your-bucket/cgrl/datasets/minari/farama/"
OUTPUT_S3 = "s3://your-bucket/cgrl/output/"
CHECKPOINT_S3 = "s3://your-bucket/cgrl/checkpoints/"

INSTANCE_TYPE = "ml.g5.xlarge"  # the only training instance the algorithm declares
JOB_PREFIX = "cgrl-train"

for name, value in [
    ("ALGORITHM_ARN", ALGORITHM_ARN),
    ("ROLE_ARN", ROLE_ARN),
    ("DATA_S3", DATA_S3),
    ("OUTPUT_S3", OUTPUT_S3),
    ("CHECKPOINT_S3", CHECKPOINT_S3),
]:
    if "<" in value or "your-" in value:
        raise ValueError(f"{name} still holds a placeholder: {value}")

REGION = ALGORITHM_ARN.split(":")[3]
session = sagemaker.Session(boto3.Session(region_name=REGION))
s3 = session.boto_session.client("s3")
sm = session.boto_session.client("sagemaker")


def split_uri(uri):
    """s3://bucket/a/b/ -> ('bucket', 'a/b/')"""
    bucket, _, key = uri.removeprefix("s3://").partition("/")
    return bucket, key


def bucket_region(bucket):
    """Region of a bucket, in the form the rest of the API uses.

    GetBucketLocation answers with a location constraint, not a region name:
    an empty one means us-east-1, and buckets created under the legacy `EU`
    constraint answer `EU` rather than `eu-west-1`.
    """
    constraint = s3.get_bucket_location(Bucket=bucket)["LocationConstraint"]
    return {None: "us-east-1", "": "us-east-1", "EU": "eu-west-1"}.get(constraint, constraint)


# Everything stays in the job's region. AWS requires that of the training input
# bucket (S3DataSource) and of the checkpoint prefix; the output prefix joins
# them so the whole setup is one region, with no cross-region transfer charges.
print(f"region: {REGION}\n")
for label, uri in (("data", DATA_S3), ("output", OUTPUT_S3), ("checkpoints", CHECKPOINT_S3)):
    bucket = split_uri(uri)[0]
    where = bucket_region(bucket)
    print(f"{label:12} {bucket:34} {where}")
    if where != REGION:
        raise ValueError(
            f"the {label} bucket {bucket} is in {where}, but the job runs in {REGION}"
        )

## 2. Get a dataset into S3

Training is bring-your-own. The job reads only what is under your `training`
channel prefix, and there is no download path out of the container — so the
first move is getting Minari dataset directories into `DATA_S3`.

This quickstart uses `mujoco/walker2d/simple-v0` from the public Farama Minari
registry: 210 MB, 1017 episodes, ~1M transitions, `Box(17)` observations and
`Box(6)` actions. Fetching it here and uploading it to your own bucket *is* the
bring-your-own path — by the time the job sees it, it is your data in your
account.

`minari download` writes `~/.minari/datasets/<namespace>/<name>/<version>/`,
which is already the tree the training channel expects:

```text
s3://your-bucket/cgrl/datasets/minari/farama/
  mujoco/
    walker2d/
      simple-v0/
        data/
          main_data.hdf5
          metadata.json
```

A dataset id is `<namespace>/<name>/<version>`, and it is also the path below
the root, so the two always match. `dataset_ids` in section 3 is resolved
against that root.

To train on your own recordings instead, package them into this same shape with
[collection/](https://github.com/ccnets-team/causal-gpt-rl/tree/main/collection),
then skip the download below and upload your own directory.

In [ ]:
import os
from pathlib import Path

import minari

DATASET_IDS = ["mujoco/walker2d/simple-v0"]

# Where minari puts downloads. This is the same default minari itself resolves.
MINARI_ROOT = Path(os.environ.get("MINARI_DATASETS_PATH", Path.home() / ".minari" / "datasets"))

for dataset_id in DATASET_IDS:
    local = MINARI_ROOT / dataset_id
    if not local.exists():
        minari.download_dataset(dataset_id)
    size_mb = sum(f.stat().st_size for f in local.rglob("*") if f.is_file()) / 1e6
    print(f"{dataset_id}: {size_mb:,.1f} MB at {local}")

In [ ]:
data_bucket, data_prefix = split_uri(DATA_S3)

# Already-uploaded objects of the same size are skipped, so this cell is safe to
# re-run without paying for the transfer twice.
uploaded = {}
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=data_bucket, Prefix=data_prefix):
    for obj in page.get("Contents", []):
        uploaded[obj["Key"]] = obj["Size"]

for dataset_id in DATASET_IDS:
    local = MINARI_ROOT / dataset_id
    for path in sorted(p for p in local.rglob("*") if p.is_file()):
        key = f"{data_prefix}{dataset_id}/{path.relative_to(local).as_posix()}"
        if uploaded.get(key) == path.stat().st_size:
            print(f"  = {key}")
            continue
        s3.upload_file(str(path), data_bucket, key)
        print(f"  + {key}")

# Read back what the job will resolve, from S3 rather than from the local copy.
for dataset_id in DATASET_IDS:
    prefix = f"{data_prefix}{dataset_id}/"
    keys = [
        obj["Key"]
        for obj in s3.list_objects_v2(Bucket=data_bucket, Prefix=prefix).get("Contents", [])
    ]
    if not any(key.endswith(".hdf5") for key in keys):
        raise FileNotFoundError(f"no .hdf5 under s3://{data_bucket}/{prefix}")
    print(f"\n{dataset_id}: {len(keys)} object(s) in S3")

## 3. Hyperparameters

`dataset_ids` is the only field you have to supply. Everything else arrives
already set, and those defaults are the recipe every published bundle was
trained with — they are spelled out below, commented, so you can see what you
are inheriting.

SageMaker passes hyperparameters as `Map<String,String>`, so every value here is
a string and lists are comma-separated. A Python list would arrive as its `repr`
and split into broken values.

`max_steps` is set low below: a short run that still reaches its archive steps,
so you can see the whole path — delivered bundles, the final artifact, the
runtime opening both — without waiting for a full training run. **It is not a
free dry run.** It bills Marketplace software and SageMaker infrastructure for as
long as it runs, exactly like any other job. Raise it to `100000` for a real one.

`0` is also accepted, if all you want to know is whether a job starts. A run
that reaches no archive step has no archive series to deliver, so section 6 will
have nothing to list.

The full field list — what is clamped, what is rounded, and what fails the job at
startup — is in
[sagemaker-inputs.md](https://github.com/ccnets-team/causal-gpt-rl/blob/main/training/docs/aws/sagemaker-inputs.md).

In [ ]:
hyperparameters = {
    "dataset_ids": ",".join(DATASET_IDS),  # required
    "max_steps": "1600",                   # short billed run; the default recipe is 100000
    # --- Training ---------------------------------------------------------
    # "batch_size": "128",        # nearest of 32/64/128/256/512
    # "context_length": "32",     # nearest of 16/24/32/48/64
    # "gamma": "0.99",            # clamped to [0.98, 0.995]
    # "td_lambda": "0.95",        # clamped to [0.90, 0.975]
    # "seed": "42",
    # --- Optimization -----------------------------------------------------
    # "learning_rate": "1e-4",    # clamped to [1e-6, 1e-3]
    # "min_lr": "1e-6",           # clamped, then capped at learning_rate
    # "lr_scheduler_type": "cosine",
    # --- Network: used exactly as given, or the job fails ------------------
    # "d_model": "256",           # divisible by num_heads
    # "num_layers": "4",
    # "num_heads": "8",
    # "dropout": "0.05",
    # --- Checkpointing ----------------------------------------------------
    # "archive_steps": "800,1600",  # extra steps preserved permanently
}

hyperparameters

## 4. Launch

`checkpoint_s3_uri` is what makes the job deliver policy bundles while it runs.
Without it the job still works — you just see nothing until it finishes. It has
to be a fresh location per job, so the job name goes into the path.

In [ ]:
import time

from sagemaker.algorithm import AlgorithmEstimator
from sagemaker.inputs import TrainingInput

job_name = f"{JOB_PREFIX}-{time.strftime('%Y%m%d-%H%M%S')}"
checkpoint_uri = f"{CHECKPOINT_S3.rstrip('/')}/{job_name}/"

estimator = AlgorithmEstimator(
    algorithm_arn=ALGORITHM_ARN,
    role=ROLE_ARN,
    instance_count=1,
    instance_type=INSTANCE_TYPE,
    output_path=OUTPUT_S3,
    checkpoint_s3_uri=checkpoint_uri,
    sagemaker_session=session,
    hyperparameters=hyperparameters,
)

# The algorithm declares one channel, one content type, one input mode. A bare
# S3 URI would also work -- the SDK checks the channel name, not the content
# type -- but naming them puts the contract in front of you.
estimator.fit(
    {"training": TrainingInput(DATA_S3, content_type="application/x-minari", input_mode="File")},
    job_name=job_name,
    wait=False,
)

print(f"job:        {job_name}")
print(f"checkpoints {checkpoint_uri}")

## 5. Watch it start

The first thing the job prints to CloudWatch Logs is a validation summary of how
your data resolved. **Read it before letting the run continue.** The line that
matters is the flattened action shape against your action space, because that is
where an encoding mistake shows: they match for a continuous `Box`, and they do
not for categorical actions — `MultiDiscrete([3, 3, 3])` is three indices, but
the flattened shape is `(9,)`, the sum of the one-hot blocks.

The cell below is re-runnable. To stream the logs instead, use
`estimator.latest_training_job.wait(logs="All")` — that call blocks until the job
ends, so the rest of this notebook waits with it.

In [ ]:
described = sm.describe_training_job(TrainingJobName=job_name)
print(described["TrainingJobStatus"], "/", described["SecondaryStatus"])
print(described.get("FailureReason", ""))

for metric in described.get("FinalMetricDataList", []):
    print(f"  {metric['MetricName']:38} {metric['Value']:.4f}")

## 6. Load a policy while the job runs

Training runs offline: there is no simulator in the container, so the job cannot
measure episode return — the thing you actually care about. That leaves one
trustworthy check, which is to run the policy in your own environment. So the
job exports runnable bundles as it goes and syncs them to the checkpoint prefix.

`archive/` points are preserved for the life of the prefix and named by step — 5
evenly spaced across `max_steps`, at 20/40/60/80/100%, plus anything you asked
for with `archive_steps`. Read the list from `manifest.json` rather than
hardcoding a count.

S3 sync order is not guaranteed. Treat a bundle that does not load as a skip and
retry it — the next delivery is unaffected either way.

In [ ]:
import json
import os

from causal_gpt_rl.inference import load_runner

ckpt_bucket, ckpt_prefix = split_uri(checkpoint_uri)


def download_prefix(bucket, prefix, dest):
    """Copy every object under `prefix` into `dest`, keeping the relative tree."""
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith("/"):
                continue
            target = os.path.join(dest, os.path.relpath(obj["Key"], prefix))
            os.makedirs(os.path.dirname(target), exist_ok=True)
            s3.download_file(bucket, obj["Key"], target)
    return dest


# The namespace level is derived from the run's dataset identity, so find the
# manifest rather than guessing the path.
manifest = None
bundles_prefix = None
manifest_keys = [
    obj["Key"]
    for page in s3.get_paginator("list_objects_v2").paginate(
        Bucket=ckpt_bucket, Prefix=ckpt_prefix
    )
    for obj in page.get("Contents", [])
    if obj["Key"].endswith("archive/bundles/manifest.json")
]

if not manifest_keys:
    print("no archive manifest yet -- re-run once the job reaches its first archive step")
else:
    bundles_prefix = manifest_keys[0].removesuffix("manifest.json")
    manifest = json.loads(s3.get_object(Bucket=ckpt_bucket, Key=manifest_keys[0])["Body"].read())
    for point in manifest["points"]:
        score = point.get("metrics", {}).get("eval_offline/checkpoint_score")
        print(f"  step {point['step']:>9}  {','.join(point['reasons']):18} score={score}")

In [ ]:
if manifest is None:
    print("nothing delivered yet -- re-run the cell above, then this one")
else:
    latest = max(point["step"] for point in manifest["points"])
    local = download_prefix(
        ckpt_bucket, f"{bundles_prefix}step_{latest:07d}/", f"./step_{latest:07d}"
    )
    # A bundle that is still syncing raises here. That is a skip, not a failure:
    # re-run this cell, or take the step below it.
    runner = load_runner(local)
    print(f"step {latest}: {runner}")

## 7. The finished model

The cell below **blocks until the job ends**, then fails loudly if it ended as
anything other than `Completed`.

`bundle/` is the canonical bundle, selected by the checkpoint metric — load this
by default. `archive/bundles/` is the same preserved series as above, included so
the run's candidates can be compared after the job ends, and `canonical.pt` is
the training state behind the canonical bundle, for resuming rather than for
inference.

The selection metric is measured on a held-out split of your dataset, not against
your environment, and the gap can be large. Treat the canonical bundle as a
default, not as a verdict — the archive points exist so you can score them
yourself and pick the one your environment prefers.

In [ ]:
import tarfile

from botocore.exceptions import WaiterError

# The artifact does not exist until the job ends. botocore polls every 120s for
# 180 attempts by default -- 6 hours; this run is short, so poll faster. Raise
# MaxAttempts if you raised max_steps.
try:
    sm.get_waiter("training_job_completed_or_stopped").wait(
        TrainingJobName=job_name,
        WaiterConfig={"Delay": 60, "MaxAttempts": 180},
    )
except WaiterError as exc:
    # A Failed job trips the waiter instead of returning from it, and the
    # waiter's own message does not carry FailureReason -- so read the job.
    print(f"waiter gave up: {exc}")

described = sm.describe_training_job(TrainingJobName=job_name)
status = described["TrainingJobStatus"]
if status != "Completed":
    hint = " (raise WaiterConfig MaxAttempts)" if status == "InProgress" else ""
    raise RuntimeError(
        f"{job_name} is {status}: {described.get('FailureReason', 'no FailureReason')}{hint}"
    )

artifact_bucket, artifact_key = split_uri(described["ModelArtifacts"]["S3ModelArtifacts"])
s3.download_file(artifact_bucket, artifact_key, "model.tar.gz")

with tarfile.open("model.tar.gz") as tar:
    try:
        tar.extractall("./model", filter="data")
    except TypeError:  # Python < 3.12
        tar.extractall("./model")

runner = load_runner("./model/bundle")
print(json.load(open("./model/reports/summary.json"))["evaluation"])

## Next

`runner` is a `PolicyRunner`: `reset(obs)`, then `act()` and `observe(obs)` each
step, in your environment's own observation and action spaces. For a Gymnasium
env, `run_episodes(env, runner, num_episodes=...)` does that loop and returns
return statistics. See the
[quick start](https://github.com/ccnets-team/causal-gpt-rl#quick-start) and
[docs/spaces.md](https://github.com/ccnets-team/causal-gpt-rl/blob/main/docs/spaces.md)
for structured spaces.

Scoring the archive points in your own environment is also how you stop a run
early: if your score is flat or falling across steps, call `StopTrainingJob` and
stop paying there rather than at `max_steps`.

```python
sm.stop_training_job(TrainingJobName=job_name)
```

To continue from the delivered model instead of starting over, see
[retraining](https://github.com/ccnets-team/causal-gpt-rl/blob/main/training/docs/aws/sagemaker-retraining.md).

A training job bills software and infrastructure separately, and both stop when
the job does. Delete anything you created for this run that you do not want to
keep paying for — the checkpoint and output prefixes hold data after the job ends.